In [ ]:
# Load R magic extension for Python Jupyter kernel (Kaggle / Colab support)
try:
    %load_ext rpy2.ipython
except Exception as e:
    print("Note on rpy2 initialization:", e)

# Train & Deploy Out-Of-Fold (OOF) Stacking Multinomial Logistic Regression Meta-Learner (`models/train_oof_logistic_regression_stacking.ipynb`)

This dedicated notebook executes a **5-Fold Out-Of-Fold (OOF) Stacking Strategy** using the **38-feature uniform input matrix** and **SMOTE upsampling**:

### Pipeline Steps
1. **5-Fold Cross-Validation OOF Generation**: Trains the 4 LightGBM sub-models across 5 folds without data leakage, generating OOF joint probability matrix $\mathbf{X}_{\text{oof}} \in \mathbb{R}^{N \times 5}$.
2. **Multinomial Logistic Regression Meta-Learner**: Fits `LogisticRegression(multi_class='multinomial', class_weight='balanced')` on $\mathbf{X}_{\text{oof}}$.
3. **Export RDS & JSON Artifacts**: Saves all 4 sub-model RDS files, full Meta-Learner RDS bundle, and JSON linear coefficients for pure C transpilation.
4. **Holdout Test Evaluation**: Detailed per-class breakdown (**Recall, Specificity, Balanced Accuracy, ROC-AUC**).

In [ ]:
%%R
# ---------------------------------------------------------
# Step 1: Load Environment & Construct 38-Feature Matrix
# ---------------------------------------------------------
suppressPackageStartupMessages({
  library(jsonlite)
  library(dplyr)
  library(ggplot2)
  library(tidyr)
  library(pROC)
  library(lightgbm)
})
config_path <- "../config/triage_conf.json"
if (!file.exists(config_path)) config_path <- "config/triage_conf.json"
config <- fromJSON(config_path)
set.seed(config$training$random_state)
stratified_partition <- function(y, p, seed = 42) {
  set.seed(seed)
  idx_list <- split(seq_along(y), y)
  train_idx <- unlist(lapply(idx_list, function(indices) {
    sample(indices, size = max(1, round(length(indices) * p)))
  }))
  return(sort(train_idx))
}
fit_scaler <- function(df_train, cols) {
  means <- colMeans(df_train[, cols, drop = FALSE], na.rm = TRUE)
  sds   <- apply(df_train[, cols, drop = FALSE], 2, sd, na.rm = TRUE)
  sds[sds == 0] <- 1
  return(list(means = means, sds = sds, cols = cols))
}
apply_scaler <- function(df, scaler) {
  df_out <- df
  for (c in scaler$cols) {
    df_out[[c]] <- (df[[c]] - scaler$means[c]) / scaler$sds[c]
  }
  return(df_out)
}
smote_binary_data <- function(df, feat_cols, label_vec, seed = 42) {
  set.seed(seed)
  pos_idx <- which(label_vec == 1)
  neg_idx <- which(label_vec == 0)
  n_pos <- length(pos_idx)
  n_neg <- length(neg_idx)
  if (n_pos == 0 || n_neg == 0 || n_pos == n_neg) return(list(X = as.matrix(df[, feat_cols]), y = label_vec))
  
  if (n_pos < n_neg) {
    minority_idx <- pos_idx
    target_syn   <- n_neg - n_pos
    min_label    <- 1
  } else {
    minority_idx <- neg_idx
    target_syn   <- n_pos - n_neg
    min_label    <- 0
  }
  
  X_min <- as.matrix(df[minority_idx, feat_cols, drop = FALSE])
  syn_matrix <- matrix(0, nrow = target_syn, ncol = length(feat_cols))
  
  for (i in 1:target_syn) {
    base_i <- sample(1:nrow(X_min), 1)
    nn_i   <- sample(1:nrow(X_min), 1)
    alpha  <- runif(1, 0, 1)
    syn_matrix[i, ] <- X_min[base_i, ] + alpha * (X_min[nn_i, ] - X_min[base_i, ])
  }
  
  X_full <- rbind(as.matrix(df[, feat_cols]), syn_matrix)
  y_full <- c(label_vec, rep(min_label, target_syn))
  return(list(X = X_full, y = y_full))
}
data_file <- config$path$data_source
if (!file.exists(data_file) && file.exists(paste0("../", data_file))) data_file <- paste0("../", data_file)
data_env <- new.env()
load(data_file, envir = data_env)
df_names <- ls(data_env)[sapply(ls(data_env), function(x) is.data.frame(get(x, envir = data_env)))]
df_sizes <- sapply(df_names, function(x) nrow(get(x, envir = data_env)))
raw_df   <- get(df_names[which.max(df_sizes)], envir = data_env)
target_col_name <- config$classes$target_col
gender_vec <- if ("gender" %in% names(raw_df)) ifelse(as.character(raw_df$gender) == "Male", 1, 0) else 0
cc_bd_vec  <- if ("cc_breathingdifficulty" %in% names(raw_df)) ifelse(!is.na(raw_df$cc_breathingdifficulty), raw_df$cc_breathingdifficulty, 0) else 0
get_vec <- function(col_name, default_val = 0) {
  if (col_name %in% names(raw_df)) {
    res <- raw_df[[col_name]]
    res[is.na(res)] <- default_val
    return(res)
  } else {
    return(rep(default_val, nrow(raw_df)))
  }
}
pulse_last <- get_vec("pulse_last"); pulse_max <- get_vec("pulse_max"); pulse_min <- get_vec("pulse_min")
sbp_last   <- get_vec("sbp_last");   sbp_max   <- get_vec("sbp_max");   sbp_min   <- get_vec("sbp_min")
spo2_last  <- get_vec("spo2_last");  spo2_max  <- get_vec("spo2_max");  spo2_min  <- get_vec("spo2_min")
resp_last  <- get_vec("resp_last");  resp_max  <- get_vec("resp_max");  resp_min  <- get_vec("resp_min")
t_hr       <- get_vec("triage_vital_hr"); t_sbp <- get_vec("triage_vital_sbp"); t_o2 <- get_vec("triage_vital_o2"); t_rr <- get_vec("triage_vital_rr")
hr_rng   <- pulse_max - pulse_min
sbp_rng  <- sbp_max - sbp_min
rr_rng   <- resp_max - resp_min
spo2_rng <- spo2_max - spo2_min
df_master <- data.frame(
  age                     = raw_df$age,
  cc_breathingdifficulty  = cc_bd_vec,
  gender                  = gender_vec,
  triage_vital_hr         = t_hr,
  triage_vital_sbp        = t_sbp,
  triage_vital_rr         = t_rr,
  triage_vital_o2         = t_o2,
  pulse_min               = pulse_min,
  resp_min                = resp_min,
  spo2_min                = spo2_min,
  sbp_min                 = sbp_min,
  pulse_max               = pulse_max,
  resp_max                = resp_max,
  spo2_max                = spo2_max,
  sbp_max                 = sbp_max,
  is_dyspnea_total        = ifelse(t_o2 < 90, 1, 0),
  is_dyspnea_moderate     = ifelse(t_o2 > 90 & t_o2 < 94, 1, 0),
  is_bradypnea            = ifelse(t_rr < 10, 1, 0),
  is_tachypnea            = ifelse(t_rr > 30, 1, 0),
  is_hypotension          = ifelse(t_sbp <= 90, 1, 0),
  is_hypertension         = ifelse(t_sbp > 220, 1, 0),
  is_bradycardia_total    = ifelse(t_hr < 40, 1, 0),
  is_bradycardia_moderate = ifelse(t_hr > 40 & t_hr < 60, 1, 0),
  is_tachycardia_total    = ifelse(t_hr > 150, 1, 0),
  is_tachycardia_moderate = ifelse(t_hr > 100 & t_hr < 150, 1, 0),
  hr_range                = hr_rng,
  rr_range                = rr_rng,
  spo2_range              = spo2_rng,
  sbp_range               = sbp_rng,
  shock_index             = t_hr / ifelse(t_sbp == 0, 1, t_sbp),
  hr_mid_to_triage        = t_hr - hr_rng,
  sbp_mid_to_triage       = t_sbp - sbp_rng,
  rr_mid_to_triage        = t_rr - rr_rng,
  spo2_mid_to_triage      = t_o2 - spo2_rng,
  rox_index               = t_o2 / ifelse(t_rr == 0, 1, t_rr),
  spo2_drop_ratio         = spo2_rng / ifelse(spo2_max == 0, 1, spo2_max),
  hr_instability_ratio    = hr_rng / (t_hr + 1),
  bif                     = (t_rr / ifelse(t_o2 == 0, 1, t_o2)) * 100
)
layer_feat_names <- names(df_master)
raw_esi <- as.character(raw_df[[target_col_name]])
df_master$target_col <- factor(raw_esi, levels = c("1", "2", "3", "4", "5"))
df_master <- na.omit(df_master)
test_size <- config$training$test_size
in_train  <- stratified_partition(df_master$target_col, p = 1 - test_size, seed = config$training$random_state)
train_df  <- df_master[in_train, ]
test_df   <- df_master[-in_train, ]
cont_cols <- c("age", "triage_vital_hr", "triage_vital_sbp", "triage_vital_rr", "triage_vital_o2", "pulse_min", "resp_min", "spo2_min", "sbp_min", "pulse_max", "resp_max", "spo2_max", "sbp_max", "hr_range", "rr_range", "spo2_range", "sbp_range", "shock_index", "hr_mid_to_triage", "sbp_mid_to_triage", "rr_mid_to_triage", "spo2_mid_to_triage", "rox_index", "spo2_drop_ratio", "hr_instability_ratio", "bif")
scaler <- fit_scaler(train_df, cont_cols)
train_scaled <- apply_scaler(train_df, scaler)
test_scaled  <- apply_scaler(test_df, scaler)
cat(sprintf("Dataset Ready: Train=%d, Holdout Test=%d with 38 Uniform Features\n", nrow(train_scaled), nrow(test_scaled)))

In [ ]:
%%R
# ---------------------------------------------------------
# Step 2: 5-Fold Cross-Validation OOF Generation & Full Model Training
# ---------------------------------------------------------
set.seed(config$training$random_state)
K <- 5
folds <- split(sample(1:nrow(train_scaled)), rep(1:K, length.out = nrow(train_scaled)))
oof_probs <- matrix(0, nrow = nrow(train_scaled), ncol = 5)
test_probs_folds <- matrix(0, nrow = nrow(test_scaled), ncol = 5)
lgb_binary_params <- list(
  objective        = "binary",
  metric           = "binary_logloss",
  learning_rate    = 0.05,
  num_leaves       = 31,
  max_depth        = 6,
  feature_fraction = 0.8,
  bagging_fraction = 0.8,
  bagging_freq     = 1,
  verbosity        = -1L
)
X_test_mat <- as.matrix(test_scaled[, layer_feat_names])
for (k in 1:K) {
  val_idx   <- folds[[k]]
  tr_idx    <- setdiff(1:nrow(train_scaled), val_idx)
  
  fold_tr  <- train_scaled[tr_idx, ]
  fold_val <- train_scaled[val_idx, ]
  
  X_val_k <- as.matrix(fold_val[, layer_feat_names])
  
  # L1
  smote_l1 <- smote_binary_data(fold_tr, layer_feat_names, ifelse(fold_tr$target_col == "1", 1, 0), seed = config$training$random_state + k)
  dtr_l1   <- lgb.Dataset(smote_l1$X, label = smote_l1$y)
  dvl_l1   <- lgb.Dataset(X_val_k, label = ifelse(fold_val$target_col == "1", 1, 0))
  l1_m     <- lgb.train(params = lgb_binary_params, data = dtr_l1, nrounds = 100, valids = list(val = dvl_l1), early_stopping_rounds = 10, verbose = -1L)
  
  # L2
  tr_l2 <- fold_tr  %>% filter(target_col != "1")
  vl_l2 <- fold_val %>% filter(target_col != "1")
  smote_l2 <- smote_binary_data(tr_l2, layer_feat_names, ifelse(tr_l2$target_col %in% c("2", "3"), 1, 0), seed = config$training$random_state + k)
  dtr_l2   <- lgb.Dataset(smote_l2$X, label = smote_l2$y)
  dvl_l2   <- lgb.Dataset(as.matrix(vl_l2[, layer_feat_names]), label = ifelse(vl_l2$target_col %in% c("2", "3"), 1, 0))
  l2_m     <- lgb.train(params = lgb_binary_params, data = dtr_l2, nrounds = 100, valids = list(val = dvl_l2), early_stopping_rounds = 10, verbose = -1L)
  
  # L3A
  tr_l3a <- fold_tr  %>% filter(target_col %in% c("2", "3"))
  vl_l3a <- fold_val %>% filter(target_col %in% c("2", "3"))
  smote_l3a <- smote_binary_data(tr_l3a, layer_feat_names, ifelse(tr_l3a$target_col == "2", 1, 0), seed = config$training$random_state + k)
  dtr_l3a   <- lgb.Dataset(smote_l3a$X, label = smote_l3a$y)
  dvl_l3a   <- lgb.Dataset(as.matrix(vl_l3a[, layer_feat_names]), label = ifelse(vl_l3a$target_col == "2", 1, 0))
  l3a_m     <- lgb.train(params = lgb_binary_params, data = dtr_l3a, nrounds = 100, valids = list(val = dvl_l3a), early_stopping_rounds = 10, verbose = -1L)
  
  # L3B
  tr_l3b <- fold_tr  %>% filter(target_col %in% c("4", "5"))
  vl_l3b <- fold_val %>% filter(target_col %in% c("4", "5"))
  smote_l3b <- smote_binary_data(tr_l3b, layer_feat_names, ifelse(tr_l3b$target_col == "4", 1, 0), seed = config$training$random_state + k)
  dtr_l3b   <- lgb.Dataset(smote_l3b$X, label = smote_l3b$y)
  dvl_l3b   <- lgb.Dataset(as.matrix(vl_l3b[, layer_feat_names]), label = ifelse(vl_l3b$target_col == "4", 1, 0))
  l3b_m     <- lgb.train(params = lgb_binary_params, data = dtr_l3b, nrounds = 100, valids = list(val = dvl_l3b), early_stopping_rounds = 10, verbose = -1L)
  
  # OOF Validation Predictions for Fold K
  p1_k  <- predict(l1_m,  X_val_k)
  p2_k  <- predict(l2_m,  X_val_k)
  p3a_k <- predict(l3a_m, X_val_k)
  p3b_k <- predict(l3b_m, X_val_k)
  
  oof_probs[val_idx, 1] <- p1_k
  oof_probs[val_idx, 2] <- (1 - p1_k) * p2_k * p3a_k
  oof_probs[val_idx, 3] <- (1 - p1_k) * p2_k * (1 - p3a_k)
  oof_probs[val_idx, 4] <- (1 - p1_k) * (1 - p2_k) * p3b_k
  oof_probs[val_idx, 5] <- (1 - p1_k) * (1 - p2_k) * (1 - p3b_k)
  
  # Test Set Accumulation across 5 Folds
  p1_ts  <- predict(l1_m,  X_test_mat)
  p2_ts  <- predict(l2_m,  X_test_mat)
  p3a_ts <- predict(l3a_m, X_test_mat)
  p3b_ts <- predict(l3b_m, X_test_mat)
  
  test_probs_folds[, 1] <- test_probs_folds[, 1] + p1_ts / K
  test_probs_folds[, 2] <- test_probs_folds[, 2] + ((1 - p1_ts) * p2_ts * p3a_ts) / K
  test_probs_folds[, 3] <- test_probs_folds[, 3] + ((1 - p1_ts) * p2_ts * (1 - p3a_ts)) / K
  test_probs_folds[, 4] <- test_probs_folds[, 4] + ((1 - p1_ts) * (1 - p2_ts) * p3b_ts) / K
  test_probs_folds[, 5] <- test_probs_folds[, 5] + ((1 - p1_ts) * (1 - p2_ts) * (1 - p3b_ts)) / K
}
# Train Final Sub-Models on 100% of Train Set for Production Deployment
smote_l1_full  <- smote_binary_data(train_scaled, layer_feat_names, ifelse(train_scaled$target_col == "1", 1, 0), seed = config$training$random_state)
full_l1_m      <- lgb.train(params = lgb_binary_params, data = lgb.Dataset(smote_l1_full$X, label = smote_l1_full$y), nrounds = 100, verbose = -1L)
tr_l2_full     <- train_scaled %>% filter(target_col != "1")
smote_l2_full  <- smote_binary_data(tr_l2_full, layer_feat_names, ifelse(tr_l2_full$target_col %in% c("2", "3"), 1, 0), seed = config$training$random_state)
full_l2_m      <- lgb.train(params = lgb_binary_params, data = lgb.Dataset(smote_l2_full$X, label = smote_l2_full$y), nrounds = 100, verbose = -1L)
tr_l3a_full    <- train_scaled %>% filter(target_col %in% c("2", "3"))
smote_l3a_full <- smote_binary_data(tr_l3a_full, layer_feat_names, ifelse(tr_l3a_full$target_col == "2", 1, 0), seed = config$training$random_state)
full_l3a_m     <- lgb.train(params = lgb_binary_params, data = lgb.Dataset(smote_l3a_full$X, label = smote_l3a_full$y), nrounds = 100, verbose = -1L)
tr_l3b_full    <- train_scaled %>% filter(target_col %in% c("4", "5"))
smote_l3b_full <- smote_binary_data(tr_l3b_full, layer_feat_names, ifelse(tr_l3b_full$target_col == "4", 1, 0), seed = config$training$random_state)
full_l3b_m     <- lgb.train(params = lgb_binary_params, data = lgb.Dataset(smote_l3b_full$X, label = smote_l3b_full$y), nrounds = 100, verbose = -1L)
# Export Production Sub-Model RDS Files to deploy/
deploy_dir <- "../deploy"
if (!dir.exists(deploy_dir)) deploy_dir <- "deploy"
if (!dir.exists(deploy_dir)) dir.create(deploy_dir, recursive = TRUE)
saveRDS(list(model = full_l1_m,  scaler = scaler, is_l1_lgb  = TRUE), file = file.path(deploy_dir, "lightgbm_layer1_esi1_model.rds"))
saveRDS(list(model = full_l2_m,  scaler = scaler, is_lgb_l2  = TRUE), file = file.path(deploy_dir, "rf_esi23_esi45_extreme_model.rds"))
saveRDS(list(model = full_l3a_m, scaler = scaler, is_lgb_l3a = TRUE), file = file.path(deploy_dir, "lightgbm_esi23_model.rds"))
saveRDS(list(model = full_l3b_m, scaler = scaler, is_lgb_l3b = TRUE), file = file.path(deploy_dir, "lightgbm_esi45_model.rds"))
y_train_act <- as.numeric(as.character(train_df$target_col))
y_test_act  <- as.numeric(as.character(test_df$target_col))
cat("5-Fold OOF Predictions & Production Sub-Models Exported to deploy/*.rds!\n")

In [ ]:
# ---------------------------------------------------------
# Step 3: Train Multinomial Logistic Regression Meta-Learner & Extract Coefficients
# ---------------------------------------------------------
import os
import json
import numpy as np
import pandas as pd
from rpy2.robjects import r
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
X_oof   = np.array(r('oof_probs'))
y_train = np.array(r('y_train_act'))
X_test  = np.array(r('test_probs_folds'))
y_test  = np.array(r('y_test_act'))
# Fit Multinomial Logistic Regression with Class Weighting
meta_logreg = LogisticRegression(
    multi_class='multinomial',
    class_weight='balanced',
    max_iter=1000,
    random_state=42
)
meta_logreg.fit(X_oof, y_train)
intercepts = meta_logreg.intercept_.tolist() # 5 values
coef_matrix = meta_logreg.coef_.tolist()     # 5x5 matrix
print("=== Multinomial Logistic Regression Meta-Learner Trained ===")
print("Intercepts:", intercepts)
print("Coefficient Matrix:")
for idx, row in enumerate(coef_matrix):
    print(f"  Class ESI_{idx+1}: {row}")
# Save JSON for Pure C Transpilation
deploy_dir = '../deploy' if os.path.exists('../deploy') else 'deploy'
os.makedirs(deploy_dir, exist_ok=True)
meta_json_data = {
    'model_type': 'oof_multinomial_logistic_regression',
    'class_weight': 'balanced',
    'features_count': 38,
    'intercepts': intercepts,
    'coef_matrix': coef_matrix
}
with open(os.path.join(deploy_dir, 'oof_multinomial_logistic_meta_learner.json'), 'w') as f:
    json.dump(meta_json_data, f, indent=2)
print(f"Meta-Learner JSON saved to {os.path.join(deploy_dir, 'oof_multinomial_logistic_meta_learner.json')}")

In [ ]:
%%R -i intercepts -i coef_matrix
# ---------------------------------------------------------
# Step 4: Export Meta-Learner Bundle RDS to deploy/
# ---------------------------------------------------------
deploy_dir <- "../deploy"
if (!dir.exists(deploy_dir)) deploy_dir <- "deploy"
if (!dir.exists(deploy_dir)) dir.create(deploy_dir, recursive = TRUE)
meta_learner_bundle <- list(
  l1_model     = full_l1_m,
  l2_model     = full_l2_m,
  l3a_model    = full_l3a_m,
  l3b_model    = full_l3b_m,
  scaler       = scaler,
  intercepts   = intercepts,
  coef_matrix  = coef_matrix,
  feature_cols = layer_feat_names
)
saveRDS(meta_learner_bundle, file = file.path(deploy_dir, "oof_multinomial_logistic_meta_learner.rds"))
cat("OOF Multinomial Logistic Meta-Learner RDS Bundle saved to deploy/oof_multinomial_logistic_meta_learner.rds successfully!\n")

In [ ]:
# ---------------------------------------------------------
# Step 5: Holdout Test Set Evaluation & Detailed Per-Class Breakdown
# ---------------------------------------------------------
preds_unweighted  = np.argmax(X_test, axis=1) + 1
preds_meta_logreg = meta_logreg.predict(X_test)
probs_meta_logreg = meta_logreg.predict_proba(X_test)
def get_per_class_breakdown(y_true, y_pred, probs, pipeline_name):
    classes = [1, 2, 3, 4, 5]
    rows = []
    recalls, specs, bal_accs, aucs = [], [], [], []
    
    for idx, cls in enumerate(classes):
        y_bin_true = (y_true == cls).astype(int)
        y_bin_pred = (y_pred == cls).astype(int)
        
        tp = np.sum((y_bin_true == 1) & (y_bin_pred == 1))
        fn = np.sum((y_bin_true == 1) & (y_bin_pred == 0))
        fp = np.sum((y_bin_true == 0) & (y_bin_pred == 1))
        tn = np.sum((y_bin_true == 0) & (y_bin_pred == 0))
        
        rec  = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0
        bal  = (rec + spec) / 2.0
        
        try:
            auc = roc_auc_score(y_bin_true, probs[:, idx])
        except Exception:
            auc = 0.0
            
        recalls.append(rec)
        specs.append(spec)
        bal_accs.append(bal)
        aucs.append(auc)
        
        rows.append({
            'Pipeline': pipeline_name,
            'Class': f'ESI_{cls}',
            'Recall': round(rec, 4),
            'Specificity': round(spec, 4),
            'Balanced_Accuracy': round(bal, 4),
            'ROC_AUC': round(auc, 4)
        })
        
    rows.append({
        'Pipeline': pipeline_name,
        'Class': 'Macro_Average',
        'Recall': round(np.mean(recalls), 4),
        'Specificity': round(np.mean(specs), 4),
        'Balanced_Accuracy': round(np.mean(bal_accs), 4),
        'ROC_AUC': round(np.mean(aucs), 4)
    })
    
    return pd.DataFrame(rows)
df_unw    = get_per_class_breakdown(y_test, preds_unweighted,  X_test,            '3Tier_Soft_Pipeline_Unweighted')
df_logreg = get_per_class_breakdown(y_test, preds_meta_logreg,probs_meta_logreg,  'OOF_Stacking_Multinomial_Logistic_Regression')
full_report_df = pd.concat([df_unw, df_logreg], ignore_index=True)
print("========================================================================================")
print("   HOLDOUT TEST SET REPORT: OOF MULTINOMIAL LOGISTIC META-LEARNER (ESI 1..5 METRICS)")
print("========================================================================================")
print(full_report_df.to_string(index=False))
print("========================================================================================\n")
reports_dir = '../reports' if os.path.exists('../reports') else 'reports'
os.makedirs(reports_dir, exist_ok=True)
full_report_df.to_csv(os.path.join(reports_dir, 'oof_multinomial_logistic_stacking_report.csv'), index=False)
print(f"Report saved to {os.path.join(reports_dir, 'oof_multinomial_logistic_stacking_report.csv')}")

In [ ]:
# ---------------------------------------------------------
# Step 6: Plot Performance Bar Chart
# ---------------------------------------------------------
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")
esi_classes_df = full_report_df[full_report_df['Class'] != 'Macro_Average']
df_melted = pd.melt(esi_classes_df, id_vars=['Pipeline', 'Class'], value_vars=['Recall', 'Specificity', 'ROC_AUC'], var_name='Metric', value_name='Score')
plt.figure(figsize=(12, 6))
ax = sns.barplot(data=df_melted, x='Class', y='Score', hue='Pipeline', palette=['#1f77b4', '#ff7f0e'])
for p in ax.patches:
    height = p.get_height()
    if height > 0:
        ax.annotate(f'{height:.2f}',
                    (p.get_x() + p.get_width() / 2., height),
                    ha='center', va='bottom',
                    fontsize=8, xytext=(0, 2),
                    textcoords='offset points')
plt.title('OOF Multinomial Logistic Regression Meta-Learner vs Unweighted 3-Tier Soft Pipeline', fontsize=12, fontweight='bold', pad=15)
plt.ylim(0, 1.15)
plt.ylabel('Score', fontsize=11)
plt.xlabel('ESI Triage Level', fontsize=11)
plt.legend(title='Pipeline', loc='upper right')
plots_dir = '../plots' if os.path.exists('../plots') else 'plots'
os.makedirs(plots_dir, exist_ok=True)
plt.tight_layout()
plt.savefig(os.path.join(plots_dir, 'oof_multinomial_logistic_stacking_performance.png'), dpi=300)
plt.show()
print(f"Plot saved to {os.path.join(plots_dir, 'oof_multinomial_logistic_stacking_performance.png')}")